In [0]:
# Databricks notebook source

# agentic-civic-resolution-app  
# Day 1 | Notebook 1: Setup & NYC 311 Ingestion
 **Goal:** Ingest NYC 311 open complaint data → Bronze Delta table

 **Dataset:** NYC Open Data — 311 Service Requests
 - API: https://data.cityofnewyork.us/resource/erm2-nwe9.json
 - Docs: https://dev.socrata.com/foundry/data.cityofnewyork.us/erm2-nwe9

## 0. Config & Catalog Setup

In [0]:
# Databricks Unity Catalog targets — edit these to match your workspace

CATALOG      = "civicops"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.nyc311_raw"
SILVER_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.complaints"

# NYC Open Data Socrata endpoint
NYC_API_BASE  = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

# How many records to pull per page (max 50000 for Socrata)
PAGE_LIMIT    = 50_000

# Total records to ingest on Day 1 (set None to pull everything — can be slow)
MAX_RECORDS   = 500_000

## 1. Create Catalog & Schemas




In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

print(f"✓ Catalog '{CATALOG}' and schemas '{BRONZE_SCHEMA}', '{SILVER_SCHEMA}' ready.")

## 2. Fetch NYC 311 Data via Socrata API

In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_nyc311_page(offset: int, limit: int = PAGE_LIMIT) -> list[dict]:
    """Fetch one page of 311 records from NYC Open Data."""
    params = {
        "$limit":  limit,
        "$offset": offset,
        "$order":  "created_date DESC",
        # Pull last 2 years for a meaningful dataset
        "$where":  f"created_date > '{(datetime.now() - timedelta(days=730)).strftime('%Y-%m-%dT00:00:00')}'"
    }
    resp = requests.get(NYC_API_BASE, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()


In [0]:

def fetch_all_records(max_records: int | None = None) -> pd.DataFrame:
    """Page through the API until max_records is reached or data is exhausted."""
    all_records = []
    offset = 0

    while True:
        limit = PAGE_LIMIT
        if max_records:
            remaining = max_records - len(all_records)
            if remaining <= 0:
                break
            limit = min(PAGE_LIMIT, remaining)

        print(f"  Fetching offset={offset:,}  limit={limit:,} ...", end=" ")
        page = fetch_nyc311_page(offset, limit)
        if not page:
            print("done (empty page).")
            break

        all_records.extend(page)
        print(f"got {len(page):,} rows  |  total so far: {len(all_records):,}")

        if len(page) < limit:
            break
        offset += limit

    return pd.DataFrame(all_records)

In [0]:
print("Starting NYC 311 ingestion...")
raw_pdf = fetch_all_records(max_records=MAX_RECORDS)
print(f"\n✓ Fetched {len(raw_pdf):,} total records.")
print(f"  Columns ({len(raw_pdf.columns)}): {list(raw_pdf.columns)}")

## 3. Write to Bronze Delta Table (raw, schema-on-read)

In [0]:
from pyspark.sql import functions as F

# Convert to Spark — keep every raw column, cast nothing yet (Bronze = raw)
raw_sdf = spark.createDataFrame(raw_pdf)

# Add ingestion metadata columns
raw_sdf = (
    raw_sdf
    .withColumn("_ingested_at",  F.current_timestamp())
    .withColumn("_source",       F.lit("nyc_open_data_311"))
    .withColumn("_ingestion_date", F.current_date())
)

# Write as Delta, partitioned by ingestion date for efficient pruning
(
    raw_sdf.write
    .format("delta")
    .mode("overwrite")                        # Day 1: full load; switch to "append" + dedup later
    .option("overwriteSchema", "true")
    .partitionBy("_ingestion_date")
    .saveAsTable(BRONZE_TABLE)
)

print(f"✓ Bronze table written: {BRONZE_TABLE}")
print(f"  Row count: {spark.table(BRONZE_TABLE).count():,}")

## 4. Quick Sanity Check

In [0]:
display(spark.table(BRONZE_TABLE).limit(5))

In [0]:
# Column coverage report
bronze_df = spark.table(BRONZE_TABLE)
total_rows = bronze_df.count()

print(f"{'Column':<40} {'Non-null %':>10}")
print("-" * 52)
for col_name in bronze_df.columns:
    non_null = bronze_df.filter(F.col(col_name).isNotNull()).count()
    pct = (non_null / total_rows) * 100
    print(f"{col_name:<40} {pct:>9.1f}%")